# Notebook 11 — VDR/RXR axis revision

## Purpose

This revision notebook addresses reviewer comments related to the interpretation and visualization of the VDR/RXR transcriptional axis in the LINCS L1000 vitamin D perturbation dataset.

The analysis is intentionally narrow and manuscript-facing. Its goal is not to infer functional vitamin D receptor activity, but to audit and summarize transcript-level responses of selected genes associated with the canonical vitamin D signaling axis.

Specifically, this notebook evaluates whether `VDR`, `CYP24A1`, `CYP27B1`, `RXRA`, `RXRB`, and `RXRG` are present in the analyzed LINCS expression matrix, summarizes their mean transcriptional effects across cell lines, and determines whether Figure 6 should be revised to include additional RXR isoforms.

## Reviewer comments addressed

This notebook supports the response to two reviewer concerns:

1. Reviewer 2 noted that stable `VDR` transcript levels do not establish functional VDR activity, because receptor activity depends on protein abundance, ligand binding, chromatin occupancy, and cofactor availability.

2. Reviewer 1 asked why `RXRB` and `RXRG` were not included in Figure 6, given that RXR is represented by multiple isoforms rather than a single gene.

## Scope of analysis

The notebook performs the following steps:

1. Load the cleaned LINCS expression matrix and signature metadata used in the manuscript.
2. Load gene annotation metadata and map gene symbols to LINCS gene identifiers.
3. Audit the presence or absence of `VDR`, `CYP24A1`, `CYP27B1`, `RXRA`, `RXRB`, and `RXRG` in the expression matrix.
4. Compute mean LINCS Level 5 z-scores for each available axis gene by cell line.
5. Compare the current Figure 6 gene set with the expanded VDR/RXR-axis gene set.
6. Provide a conservative interpretation for manuscript revision, explicitly distinguishing transcript-level modulation from functional receptor activity.

## Output policy

No tables or figures are saved by default.

A revised Figure 6 will only be saved if the expanded VDR/RXR-axis audit supports replacing the current manuscript figure. A table will only be saved if it is promoted to supplementary material or needed for the reviewer response.

## Interpretation boundary

All values in this notebook represent transcript-level modulation derived from LINCS Level 5 moderated z-scores. They should not be interpreted as direct evidence of VDR protein abundance, receptor activation, chromatin binding, cofactor recruitment, or functional receptor activity.

---

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# Locate project root from the current notebook working directory.
def find_project_root(start_path: Path | None = None) -> Path:
    start_path = Path.cwd().resolve() if start_path is None else start_path.resolve()

    for candidate in [start_path, *start_path.parents]:
        expected_file = candidate / "data" / "exports" / "expression_matrix_clean.parquet"
        if expected_file.exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate project root. Expected to find "
        "'data/exports/expression_matrix_clean.parquet' in the current path or its parents."
    )


PROJECT_ROOT = find_project_root()

EXPORTS_DIR = PROJECT_ROOT / "data" / "exports"
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw_data"

EXPRESSION_PATH = EXPORTS_DIR / "expression_matrix_clean.parquet"
METADATA_PATH = EXPORTS_DIR / "signature_metadata_clean.csv"
GENEINFO_PATH = RAW_DATA_DIR / "geneinfo_beta.txt"

required_paths = {
    "expression_matrix": EXPRESSION_PATH,
    "signature_metadata": METADATA_PATH,
    "geneinfo": GENEINFO_PATH,
}

missing_paths = [name for name, path in required_paths.items() if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing required input files: {missing_paths}")

exp_df = pd.read_parquet(EXPRESSION_PATH)
sig_meta = pd.read_csv(METADATA_PATH)
geneinfo = pd.read_csv(GENEINFO_PATH, sep="\t", dtype={"gene_id": str})

exp_df.index = exp_df.index.astype(str)

required_metadata_columns = {"sig_id", "cell_id"}
missing_metadata_columns = required_metadata_columns.difference(sig_meta.columns)
if missing_metadata_columns:
    raise ValueError(f"Missing required metadata columns: {sorted(missing_metadata_columns)}")

required_geneinfo_columns = {"gene_id", "gene_symbol"}
missing_geneinfo_columns = required_geneinfo_columns.difference(geneinfo.columns)
if missing_geneinfo_columns:
    raise ValueError(f"Missing required geneinfo columns: {sorted(missing_geneinfo_columns)}")

metadata_sig_ids = set(sig_meta["sig_id"])
expression_sig_ids = set(exp_df.columns)

missing_from_expression = sorted(metadata_sig_ids.difference(expression_sig_ids))
missing_from_metadata = sorted(expression_sig_ids.difference(metadata_sig_ids))

if missing_from_expression or missing_from_metadata:
    raise ValueError(
        "Expression matrix and metadata are not fully aligned by sig_id. "
        f"Missing from expression: {len(missing_from_expression)}; "
        f"missing from metadata: {len(missing_from_metadata)}."
    )

sig_meta = (
    sig_meta
    .set_index("sig_id")
    .loc[list(exp_df.columns)]
    .rename_axis("sig_id")
    .reset_index()
)

print("Input dimensions")
print("-" * 40)
print(f"Expression matrix:     {exp_df.shape[0]:,} genes x {exp_df.shape[1]:,} signatures")
print(f"Signature metadata:    {sig_meta.shape[0]:,} signatures x {sig_meta.shape[1]:,} columns")
print(f"Gene annotation table: {geneinfo.shape[0]:,} genes x {geneinfo.shape[1]:,} columns")
print("")
print("Alignment checks")
print("-" * 40)
print(f"Metadata aligned to expression columns: {sig_meta['sig_id'].tolist() == list(exp_df.columns)}")
print(f"Unique expression genes:               {exp_df.index.is_unique}")
print(f"Unique metadata signatures:            {sig_meta['sig_id'].is_unique}")
print(f"Unique geneinfo gene IDs:              {geneinfo['gene_id'].is_unique}")

In [ ]:
AXIS_GENES = ["VDR", "CYP24A1", "CYP27B1", "RXRA", "RXRB", "RXRG"]
CURRENT_FIGURE6_GENES = ["CYP24A1", "CYP27B1", "RXRA", "VDR"]

geneinfo_axis = geneinfo.copy()
geneinfo_axis["gene_id"] = geneinfo_axis["gene_id"].astype(str)
geneinfo_axis["gene_symbol"] = geneinfo_axis["gene_symbol"].astype(str)

expression_gene_ids = set(exp_df.index.astype(str))

audit_rows = []

for gene_symbol in AXIS_GENES:
    matches = geneinfo_axis.loc[
        geneinfo_axis["gene_symbol"].eq(gene_symbol)
    ].copy()

    gene_ids_in_geneinfo = sorted(matches["gene_id"].dropna().astype(str).unique())
    gene_ids_in_expression = [
        gene_id for gene_id in gene_ids_in_geneinfo
        if gene_id in expression_gene_ids
    ]

    feature_space_values = "not_available"
    if "feature_space" in matches.columns and not matches.empty:
        feature_space_values = ";".join(
            sorted(matches["feature_space"].dropna().astype(str).unique())
        )

    audit_rows.append(
        {
            "gene_symbol": gene_symbol,
            "n_geneinfo_matches": len(gene_ids_in_geneinfo),
            "gene_ids_in_geneinfo": ";".join(gene_ids_in_geneinfo) if gene_ids_in_geneinfo else "not_found",
            "present_in_expression": len(gene_ids_in_expression) > 0,
            "gene_ids_in_expression": ";".join(gene_ids_in_expression) if gene_ids_in_expression else "not_found",
            "feature_space": feature_space_values,
        }
    )

axis_gene_audit = pd.DataFrame(audit_rows)

available_axis_genes = axis_gene_audit.loc[
    axis_gene_audit["present_in_expression"],
    "gene_symbol"
].tolist()

missing_axis_genes = axis_gene_audit.loc[
    ~axis_gene_audit["present_in_expression"],
    "gene_symbol"
].tolist()

axis_gene_id_map = (
    axis_gene_audit
    .loc[axis_gene_audit["present_in_expression"], ["gene_symbol", "gene_ids_in_expression"]]
    .set_index("gene_symbol")["gene_ids_in_expression"]
    .to_dict()
)

print("VDR/RXR-axis gene presence audit")
print("-" * 60)
print(axis_gene_audit.to_string(index=False))
print("")
print("Summary")
print("-" * 60)
print(f"Requested axis genes:                 {len(AXIS_GENES)}")
print(f"Available in expression matrix:       {len(available_axis_genes)}")
print(f"Missing from expression matrix:       {len(missing_axis_genes)}")
print(f"Available genes:                      {available_axis_genes}")
print(f"Missing genes:                        {missing_axis_genes}")
print("")
print("Figure 6 audit")
print("-" * 60)
print(f"Current Figure 6 genes:               {CURRENT_FIGURE6_GENES}")
print(f"Expanded VDR/RXR-axis genes available:{available_axis_genes}")
print(f"RXRB available for Figure 6:          {'RXRB' in available_axis_genes}")
print(f"RXRG available for Figure 6:          {'RXRG' in available_axis_genes}")

In [ ]:
cell_order_reference = ["A549", "HA1E", "MCF7", "PC3", "U2OS"]
cell_order = [cell for cell in cell_order_reference if cell in sig_meta["cell_id"].unique()]

axis_expression_rows = []

for gene_symbol in AXIS_GENES:
    gene_ids = axis_gene_id_map.get(gene_symbol, "").split(";")
    gene_ids = [gene_id for gene_id in gene_ids if gene_id in exp_df.index]

    if len(gene_ids) == 0:
        continue

    if len(gene_ids) > 1:
        raise ValueError(
            f"Multiple expression gene IDs found for {gene_symbol}: {gene_ids}. "
            "Manual disambiguation is required before summarizing effects."
        )

    gene_id = gene_ids[0]
    values = exp_df.loc[gene_id, sig_meta["sig_id"]].astype(float)

    gene_frame = pd.DataFrame(
        {
            "sig_id": sig_meta["sig_id"].values,
            "cell_id": sig_meta["cell_id"].values,
            "gene_symbol": gene_symbol,
            "gene_id": gene_id,
            "z_score": values.values,
        }
    )

    axis_expression_rows.append(gene_frame)

axis_expression_long = pd.concat(axis_expression_rows, ignore_index=True)

axis_effects_by_cell = (
    axis_expression_long
    .groupby(["gene_symbol", "cell_id"], observed=True)["z_score"]
    .mean()
    .unstack("cell_id")
    .reindex(index=AXIS_GENES, columns=cell_order)
)

axis_effects_summary = (
    axis_expression_long
    .groupby("gene_symbol", observed=True)
    .agg(
        gene_id=("gene_id", "first"),
        mean_effect=("z_score", "mean"),
        mean_abs_effect=("z_score", lambda x: x.abs().mean()),
        min_effect=("z_score", "min"),
        max_effect=("z_score", "max"),
        n_signatures=("z_score", "size"),
    )
    .reindex(AXIS_GENES)
)

axis_signature_counts = (
    sig_meta
    .groupby("cell_id", observed=True)
    .size()
    .reindex(cell_order)
)

print("Signature counts by cell line")
print("-" * 60)
print(axis_signature_counts.to_string())
print("")
print("Mean transcript-level effects by cell line")
print("-" * 60)
print(axis_effects_by_cell.round(4).to_string())
print("")
print("Axis gene effect summary across all signatures")
print("-" * 60)
print(axis_effects_summary.round(4).to_string())
print("")
print("Current versus expanded Figure 6 gene coverage")
print("-" * 60)
print(f"Current Figure 6 genes:  {CURRENT_FIGURE6_GENES}")
print(f"Expanded Figure 6 genes: {AXIS_GENES}")

In [ ]:
figure6_decision_rows = []

for gene_symbol in AXIS_GENES:
    effects = axis_effects_by_cell.loc[gene_symbol].dropna()

    positive_cells = effects[effects > 0].index.tolist()
    negative_cells = effects[effects < 0].index.tolist()

    max_abs_cell = effects.abs().idxmax()
    max_abs_value = effects.loc[max_abs_cell]

    figure6_decision_rows.append(
        {
            "gene_symbol": gene_symbol,
            "currently_in_figure6": gene_symbol in CURRENT_FIGURE6_GENES,
            "available_in_expression": gene_symbol in available_axis_genes,
            "mean_across_cells": effects.mean(),
            "mean_abs_across_cells": effects.abs().mean(),
            "min_cell_mean": effects.min(),
            "max_cell_mean": effects.max(),
            "n_positive_cell_means": len(positive_cells),
            "n_negative_cell_means": len(negative_cells),
            "cell_with_largest_abs_effect": max_abs_cell,
            "largest_abs_effect_value": max_abs_value,
            "positive_cell_means": ",".join(positive_cells) if positive_cells else "none",
            "negative_cell_means": ",".join(negative_cells) if negative_cells else "none",
        }
    )

figure6_decision_summary = pd.DataFrame(figure6_decision_rows)

rxr_isoform_summary = figure6_decision_summary.loc[
    figure6_decision_summary["gene_symbol"].isin(["RXRA", "RXRB", "RXRG"])
].copy()

omitted_available_genes = [
    gene for gene in AXIS_GENES
    if gene not in CURRENT_FIGURE6_GENES and gene in available_axis_genes
]

print("Figure 6 revision decision summary")
print("-" * 80)
print(figure6_decision_summary.round(4).to_string(index=False))
print("")
print("RXR isoform-specific summary")
print("-" * 80)
print(rxr_isoform_summary.round(4).to_string(index=False))
print("")
print("Decision support")
print("-" * 80)
print(f"Current Figure 6 gene count:            {len(CURRENT_FIGURE6_GENES)}")
print(f"Expanded available axis gene count:     {len(AXIS_GENES)}")
print(f"Available genes omitted from Figure 6:  {omitted_available_genes}")
print("")
print("Recommended figure action")
print("-" * 80)
if omitted_available_genes:
    print(
        "Revise Figure 6 to include the available omitted RXR isoforms "
        "so that RXRA, RXRB, and RXRG are handled consistently."
    )
else:
    print(
        "No additional VDR/RXR-axis genes are available for inclusion based on the current audit."
    )

print("")
print("Interpretation boundary")
print("-" * 80)
print(
    "These summaries describe transcript-level modulation only. "
    "They do not measure VDR/RXR protein abundance, ligand-dependent receptor activation, "
    "chromatin occupancy, cofactor recruitment, or functional receptor activity."
)

In [ ]:
FIGURE6_EXPANDED_GENE_ORDER = ["CYP24A1", "CYP27B1", "RXRA", "RXRB", "RXRG", "VDR"]

figure6_matrix = (
    axis_effects_by_cell
    .reindex(index=FIGURE6_EXPANDED_GENE_ORDER, columns=cell_order)
)

if figure6_matrix.isna().any().any():
    missing_entries = figure6_matrix.isna()
    raise ValueError(
        "Figure 6 matrix contains missing values. "
        f"Missing positions:\n{missing_entries}"
    )

figure6_plot_matrix = figure6_matrix.T

max_abs_value = float(np.nanmax(np.abs(figure6_plot_matrix.values)))
norm = TwoSlopeNorm(vmin=-max_abs_value, vcenter=0, vmax=max_abs_value)

fig, ax = plt.subplots(figsize=(7.0, 3.8))

im = ax.imshow(
    figure6_plot_matrix.values,
    aspect="auto",
    cmap="coolwarm",
    norm=norm,
)

ax.set_xticks(np.arange(figure6_plot_matrix.shape[1]))
ax.set_xticklabels(figure6_plot_matrix.columns, rotation=45, ha="right")

ax.set_yticks(np.arange(figure6_plot_matrix.shape[0]))
ax.set_yticklabels(figure6_plot_matrix.index)

ax.set_title("Transcript-level changes in selected VDR/RXR-axis genes")
ax.set_xlabel("Gene")
ax.set_ylabel("Cell line")

for row_idx in range(figure6_plot_matrix.shape[0]):
    for col_idx in range(figure6_plot_matrix.shape[1]):
        value = figure6_plot_matrix.iloc[row_idx, col_idx]
        ax.text(
            col_idx,
            row_idx,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=8,
        )

colorbar = fig.colorbar(im, ax=ax)
colorbar.set_label("Mean LINCS Level 5 z-score")

fig.tight_layout()

print("Expanded Figure 6 preview generated")
print("-" * 60)
print(f"Genes shown:      {FIGURE6_EXPANDED_GENE_ORDER}")
print(f"Cell lines shown: {cell_order}")
print(f"Value range:      {figure6_plot_matrix.values.min():.4f} to {figure6_plot_matrix.values.max():.4f}")
print(f"Centered scale:   {-max_abs_value:.4f} to {max_abs_value:.4f}")

plt.show()

## Interpretation of the VDR/RXR-axis audit

All six selected VDR/RXR-axis genes were detected in the LINCS expression matrix: `VDR`, `CYP24A1`, `CYP27B1`, `RXRA`, `RXRB`, and `RXRG`. Therefore, the absence of `RXRB` and `RXRG` from the original Figure 6 was not due to lack of gene availability in the analyzed expression space.

This audit supports revising Figure 6 to include all three RXR isoforms (`RXRA`, `RXRB`, and `RXRG`) together with `VDR`, `CYP24A1`, and `CYP27B1`. This revision directly addresses the reviewer concern that RXR was treated as a single-axis component despite the existence of multiple RXR isoforms.

At the transcript level, `VDR` showed small mean effects across cell lines, with weak negative average values in A549, HA1E, PC3, and U2OS, and a weak positive value in MCF7. `CYP27B1` also remained close to baseline across most contexts. `CYP24A1` showed modest context-dependent positive modulation, most evident in HA1E and A549, but it was not uniformly induced across all cell lines.

The RXR isoforms showed distinct transcript-level patterns. `RXRA` displayed positive mean effects in HA1E, MCF7, PC3, and U2OS, with a negative mean effect in A549. `RXRB` showed mixed and generally modest effects across cell lines. `RXRG` showed negative mean effects across all five cell lines, with the strongest negative value observed in MCF7 and a near-baseline negative value in PC3.

These results should be interpreted strictly as transcript-level modulation of selected VDR/RXR-axis genes. They do not provide evidence for functional VDR/RXR receptor activity, receptor protein abundance, ligand-dependent activation, chromatin occupancy, cofactor recruitment, or downstream receptor functionality. Accordingly, manuscript language should be revised to avoid claims that stable `VDR` transcript levels imply preserved receptor pools or explain functional receptor activity. The analysis supports only the narrower conclusion that variation in downstream transcriptional responses is not accompanied by large transcript-level changes in `VDR` itself, while selected RXR isoforms show modest and context-dependent transcript-level variation.


In [ ]:
FIGURE6_OUTPUT_DIR = PROJECT_ROOT / "results" / "revision" / "figures"
FIGURE6_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIGURE6_OUTPUT_PATH = FIGURE6_OUTPUT_DIR / "figure6_vdr_rxr_axis_revised.svg"

fig, ax = plt.subplots(figsize=(7.0, 3.8))

im = ax.imshow(
    figure6_plot_matrix.values,
    aspect="auto",
    cmap="coolwarm",
    norm=norm,
)

ax.set_xticks(np.arange(figure6_plot_matrix.shape[1]))
ax.set_xticklabels(figure6_plot_matrix.columns, rotation=45, ha="right")

ax.set_yticks(np.arange(figure6_plot_matrix.shape[0]))
ax.set_yticklabels(figure6_plot_matrix.index)

ax.set_title("Transcript-level changes in selected VDR/RXR-axis genes")
ax.set_xlabel("Gene")
ax.set_ylabel("Cell line")

for row_idx in range(figure6_plot_matrix.shape[0]):
    for col_idx in range(figure6_plot_matrix.shape[1]):
        value = figure6_plot_matrix.iloc[row_idx, col_idx]
        ax.text(
            col_idx,
            row_idx,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=8,
        )

colorbar = fig.colorbar(im, ax=ax)
colorbar.set_label("Mean LINCS Level 5 z-score")

fig.tight_layout()
fig.savefig(FIGURE6_OUTPUT_PATH, bbox_inches="tight")
plt.close(fig)

print("Revised Figure 6 saved")
print("-" * 60)
print(f"Output file: {FIGURE6_OUTPUT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Genes shown: {list(figure6_plot_matrix.columns)}")
print(f"Cell lines:  {list(figure6_plot_matrix.index)}")